# Custom Weight-Only Quantizer for Model: Qwen3-0.6B

Post-training, weight-only quantization of Qwen/Qwen3-0.6B (~0.6 B parameters), written from scratch in raw PyTorch.
Weights are stored as INT8 / INT4 and rebuilt in 16-bit precision at run time (W8A16 / W4A16).

## Constraint
| # | Requirement |
|---|---|
| 1 | **Device:** laptop with 16 GB RAM, of which ~8 GB is usable after the OS. The weights must fit comfortably. |
| 2 | **Size:** total weight storage at least **60 % smaller than FP32** (parameters × 4 bytes). The reduction versus the native 16-bit checkpoint is reported as well. |
| 3 | **Quality:** WikiText-2 perplexity at most **10 % higher** than the 16-bit baseline. |

**Selection rule:** among configurations meeting 2 and 3, take the smallest (sizes within 1 % are a tie, broken by lowest perplexity). If none meets 3, take the lowest-perplexity configuration that meets 2.

## Configurations
| ID | Bits | Scales | Mode | Purpose |
|:--:|:----:|:------:|:----:|---|
| **A** | 8 | per tensor | symmetric | Baseline and "what fails" case: one scale for the whole matrix, so one outlier coarsens every weight |
| **B** | 8 | per channel | symmetric | Course standard setup: one scale per output row |
| **C** | 4 | per channel | asymmetric | Higher compression: tests 4-bit packing and a non-zero zero point |


## Stage 1 — Environment

In [ ]:
!pip install -q "transformers>=4.51.0" accelerate safetensors psutil pandas datasets

import os, time, gc, math, shutil, warnings
import psutil, torch, transformers
import torch.nn.functional as F
import pandas as pd
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

cuda_available = torch.cuda.is_available()
gpu = torch.cuda.get_device_name(0) if cuda_available else "no GPU (a GPU runtime is recommended)"
print(f"PyTorch {torch.__version__} | Transformers {transformers.__version__} | {gpu} | "
      f"RAM {psutil.virtual_memory().total / 1024**3:.1f} GB")

PyTorch 2.11.0+cu130 | Transformers 5.20.1 | Tesla T4 | RAM 12.7 GB


## Stage 2 — Baseline model
Loaded in 16-bit (BF16 where supported, otherwise FP16). All sizes are in decimal megabytes (1 MB = 1,000,000 bytes).

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"
device = "cuda" if cuda_available else "cpu"
if cuda_available and not torch.cuda.is_bf16_supported():
    dtype, BASE_NAME = torch.float16, "FP16"
else:
    dtype, BASE_NAME = torch.bfloat16, "BF16"
MB = 1e6

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
baseline_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map=device).eval()

n_params = sum(p.numel() for p in baseline_model.parameters())      # tied weights counted once
FP32_BYTES = n_params * 4
print(f"{n_params:,} parameters | FP32 {FP32_BYTES / MB:,.0f} MB | {BASE_NAME} {n_params * 2 / MB:,.0f} MB | "
      f"tied embeddings: {baseline_model.config.tie_word_embeddings}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

596,049,920 parameters | FP32 2,384 MB | BF16 1,192 MB | tied embeddings: True


## Stage 3 — Quantization mathematics
`w_min`, `w_max` are the minimum and maximum of the **original tensor**; `[q_min, q_max]` is the **quantization range** of the integer type
(`q_min = −2^(b−1)`, `q_max = 2^(b−1) − 1`: INT8 → [−128, 127], INT4 → [−8, 7]).


In [ ]:
%%writefile quantizer_core.py

import os
import json
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F


def get_qrange(bits, signed=True):
    if signed:
        return -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    return 0, 2 ** bits - 1


def calculate_qparams(w_min, w_max, q_min, q_max, symmetric=False,
                      scale_dtype=None, min_scale=1e-8):
    if symmetric:
        max_abs = torch.maximum(torch.abs(w_min), torch.abs(w_max))
        scale = max_abs / float(q_max)
    else:
        # Ensure zero is included in the representation range
        w_min = torch.clamp(w_min, max=0.0)
        w_max = torch.clamp(w_max, min=0.0)
        scale = (w_max - w_min) / float(q_max - q_min)

    scale = torch.clamp(scale, min=min_scale)
    if scale_dtype is not None:
        scale = scale.to(scale_dtype).to(w_min.dtype)

    if symmetric:
        zero_point = torch.zeros_like(scale, dtype=torch.int32)
    else:
        zero_point = torch.clamp(torch.round(q_min - w_min / scale), q_min, q_max).to(torch.int32)

    return scale, zero_point


def quantize_tensor(tensor, scale, zero_point, q_min, q_max, target_dtype=torch.int8):
    """Quantize floating point tensor to integer grid: q = clamp(round(x / scale + zero_point), q_min, q_max)."""
    scaled = tensor / scale + zero_point
    rounded = torch.round(scaled)
    clamped = torch.clamp(rounded, q_min, q_max)
    return clamped.to(target_dtype)


def dequantize_tensor(q, scale, zero_point, target_dtype=torch.float32):
    """Dequantize integer tensor back to float: x_hat = scale * (q - zero_point)."""
    return scale.to(target_dtype) * (q.to(target_dtype) - zero_point.to(target_dtype))


def calculate_error(original, reconstructed):
    """Compute Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) between tensors."""
    diff = original.to(torch.float32) - reconstructed.to(torch.float32)
    return {
        "mae": torch.mean(torch.abs(diff)).item(),
        "rmse": torch.sqrt(torch.mean(diff ** 2)).item()
    }


def quantize_weight(w, bits, granularity, symmetric=True,
                    scale_dtype=torch.float16, min_scale=2.0 ** -14):
    assert w.dtype == torch.float32, "cast weights to float32 BEFORE quantizing"
    out_f, in_f = w.shape
    q_min, q_max = get_qrange(bits)

    if granularity == "tensor":
        w_min = torch.min(w).reshape(1, 1)
        w_max = torch.max(w).reshape(1, 1)
    elif granularity == "channel":
        w_min = torch.min(w, dim=-1, keepdim=True).values
        w_max = torch.max(w, dim=-1, keepdim=True).values
    else:
        raise ValueError(f"unknown granularity '{granularity}'; must be 'tensor' or 'channel'")

    scale, zero_point = calculate_qparams(
        w_min, w_max, q_min, q_max, symmetric,
        scale_dtype=scale_dtype, min_scale=min_scale
    )
    q = quantize_tensor(w, scale, zero_point, q_min, q_max, torch.int8)

    return {
        "q": q,
        "scale": scale,
        "zero_point": zero_point,
        "w_min": w_min,
        "w_max": w_max,
        "q_min": q_min,
        "q_max": q_max
    }

Writing quantizer_core.py


In [ ]:
import importlib, quantizer_core
importlib.reload(quantizer_core)
from quantizer_core import *

def trace(values, bits, symmetric):
    w = torch.tensor(values, dtype=torch.float32)
    q_min, q_max = get_qrange(bits)
    w_min, w_max = torch.min(w), torch.max(w)
    s, z = calculate_qparams(w_min, w_max, q_min, q_max, symmetric)
    q = quantize_tensor(w, s, z, q_min, q_max)
    x_hat = dequantize_tensor(q, s, z)
    return dict(w=w, w_min=w_min.item(), w_max=w_max.item(), q_min=q_min, q_max=q_max,
                s=s, z=z, q=q, x_hat=x_hat, err=calculate_error(w, x_hat))

# The same four numbers under three settings
DEMO = [-0.60, 0.00, 0.25, 0.90]
rows = []
for label, bits, sym in [("INT8 asymmetric", 8, False), ("INT8 symmetric", 8, True), ("INT4 asymmetric", 4, False)]:
    t = trace(DEMO, bits, sym)
    rows.append({"setting": label, "original min": t["w_min"], "original max": t["w_max"],
                 "q_min": t["q_min"], "q_max": t["q_max"], "scale": round(t["s"].item(), 6),
                 "zero point": t["z"].item(), "stored q": t["q"].tolist(), "MAE": round(t["err"]["mae"], 6)})
display(pd.DataFrame(rows).set_index("setting"))

# Step-by-step on the course's worked example (INT8, asymmetric)
t = trace([-1.0, 0.0, 2.0, 10.0], 8, False)
s, z = t["s"], t["z"]
print(f"\nCourse example  min {t['w_min']}, max {t['w_max']}, range [{t['q_min']}, {t['q_max']}], scale {s.item():.4f}, zero point {z.item()}")
print("x            :", t["w"].tolist())
print("x / S + Z    :", [round(v, 3) for v in (t["w"] / s + z).tolist()])
print("q (rounded)  :", t["q"].tolist())
print("S * (q - Z)  :", [round(v, 3) for v in t["x_hat"].tolist()])
s_sym, _ = calculate_qparams(torch.tensor(-1.0), torch.tensor(10.0), -128, 127, symmetric=True)
ok = (abs(s.item() - 0.0431) < 5e-5 and z.item() == -105 and t["q"].tolist() == [-128, -105, -59, 127]
      and t["x_hat"][1].item() == 0.0 and abs(s_sym.item() - 0.0787) < 5e-5)
print(f"Matches the course values (scale 0.0431, zero point -105, q -128/-105/-59/127, exact zero, symmetric scale 0.0787): {ok}")

,original min,original max,q_min,q_max,scale,zero point,stored q,MAE
setting,,,,,,,,
INT8 asymmetric,-0.6,0.9,-128,127,0.005882,-26,"[-128, -26, 16, 127]",0.000735
INT8 symmetric,-0.6,0.9,-128,127,0.007087,0,"[-85, 0, 35, 127]",0.001083
INT4 asymmetric,-0.6,0.9,-8,7,0.100000,-2,"[-8, -2, 0, 7]",0.012500



Course example  min -1.0, max 10.0, range [-128, 127], scale 0.0431, zero point -105
x            : [-1.0, 0.0, 2.0, 10.0]
x / S + Z    : [-128.182, -105.0, -58.636, 126.818]
q (rounded)  : [-128, -105, -59, 127]
S * (q - Z)  : [-0.992, 0.0, 1.984, 10.008]
Matches the course values (scale 0.0431, zero point -105, q -128/-105/-59/127, exact zero, symmetric scale 0.0787): True


## Stage 4 — INT4 bit packing
The smallest storage unit is 8 bits, so a 4-bit value kept in an `int8` tensor saves nothing.
Each value is shifted from `[-8, 7]` to `[0, 15]` and **two values share one byte**: the even column in the low 4 bits, the odd column shifted four places left.

In [ ]:
%%writefile -a quantizer_core.py

def pack_int4(q):
    assert q.shape[1] % 2 == 0, "number of columns must be even"
    assert int(q.min()) >= -8 and int(q.max()) <= 7, "values must fit in signed 4 bits"
    u = q.to(torch.int32) + 8
    low, high = u[:, 0::2], u[:, 1::2]
    return (low | (high << 4)).to(torch.uint8)


def unpack_int4(packed, n_cols):
    p = packed.to(torch.int32)
    low = (p & 0x0F) - 8
    high = ((p >> 4) & 0x0F) - 8
    return torch.stack([low, high], dim=-1).reshape(packed.shape[0], n_cols).to(torch.int8)

Appending to quantizer_core.py


In [ ]:
importlib.reload(quantizer_core)
from quantizer_core import *

test_mat = torch.randint(-8, 8, (4, 8), dtype=torch.int8)
packed = pack_int4(test_mat)
assert torch.equal(unpack_int4(packed, 8), test_mat), "bit packing failed"
row = torch.tensor([[1, -2, 7, -8]], dtype=torch.int8)
byte_pair = pack_int4(row)
print(f"Pack/unpack is bit-exact: {test_mat.numel()} values -> {packed.numel()} bytes ({packed.numel() / test_mat.numel():.1f} bytes per value)")
print(f"Example {row.tolist()[0]} -> {[format(b, '08b') for b in byte_pair[0].tolist()]} -> {unpack_int4(byte_pair, 4).tolist()[0]}")

Pack/unpack is bit-exact: 32 values -> 16 bytes (0.5 bytes per value)
Example [1, -2, 7, -8] -> ['01101001', '00001111'] -> [1, -2, 7, -8]


## Stage 5 — Quantized layer and model transformation
`QuantizedLinear` replaces `nn.Linear`. It stores the integer weights (packed for 4-bit), float16 scales and, in asymmetric mode, int8 zero points; at run time it rebuilds the weights and calls `F.linear`.
`quantize_model` quantizes the 7 projection matrices in each of the 28 blocks (q, k, v, o, gate, up, down → **196 matrices**).
The embedding / output layer (tied in Qwen3) and the norm layers stay in 16-bit, as the course advises for sensitive layers.
Sizes are computed from the real tensors, not from formulas.

In [ ]:
%%writefile -a quantizer_core.py

class QuantizedLinear(nn.Module):
    def __init__(self, in_features, out_features, bits, granularity,
                 symmetric=True, has_bias=False):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.bits, self.granularity = bits, granularity
        self.symmetric = symmetric

        s0 = 1 if granularity == "tensor" else out_features

        if bits == 8:
            self.register_buffer("weight_q", torch.zeros(out_features, in_features, dtype=torch.int8))
        elif bits == 4:
            self.register_buffer("weight_q", torch.zeros(out_features, in_features // 2, dtype=torch.uint8))
        else:
            raise ValueError("only 8-bit and 4-bit storage implemented")

        self.register_buffer("scale", torch.ones(s0, 1, dtype=torch.float16))

        if symmetric:
            self.register_buffer("zero_point", None)
        else:
            self.register_buffer("zero_point", torch.zeros(s0, 1, dtype=torch.int8))

        if has_bias:
            self.register_buffer("bias", torch.zeros(out_features, dtype=torch.bfloat16))
        else:
            self.register_buffer("bias", None)

    @classmethod
    def from_float(cls, linear, bits, granularity, symmetric=True):
        """Construct a QuantizedLinear module from a standard float Linear layer."""
        w32 = linear.weight.data.to(torch.float32)
        r = quantize_weight(w32, bits, granularity, symmetric=symmetric)
        layer = cls(linear.in_features, linear.out_features, bits, granularity,
                    symmetric, linear.bias is not None).to(linear.weight.device)
        layer.weight_q.copy_(pack_int4(r["q"]) if bits == 4 else r["q"])
        layer.scale.copy_(r["scale"].to(torch.float16))
        if not symmetric:
            layer.zero_point.copy_(r["zero_point"].to(torch.int8))
        if linear.bias is not None:
            layer.bias.copy_(linear.bias.data.to(layer.bias.dtype))
        return layer

    def dequantize_weight(self, dtype=torch.float32):
        """Reconstruct float weights from integer buffer, scale, and zero-point."""
        q = unpack_int4(self.weight_q, self.in_features) if self.bits == 4 else self.weight_q
        w = q.to(torch.float32)
        if self.zero_point is not None:
            w = w - self.zero_point.to(torch.float32)
        w = w * self.scale.to(torch.float32)
        return w.to(dtype)

    def forward(self, x):
        w = self.dequantize_weight(x.dtype)
        b = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w, b)

    def extra_repr(self):
        return (f"in={self.in_features}, out={self.out_features}, bits={self.bits}, "
                f"granularity={self.granularity}, symmetric={self.symmetric}")


TARGET_PROJECTIONS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


def _target_linears(model):
    return [(n, m) for n, m in model.named_modules()
            if isinstance(m, nn.Linear) and n.split(".")[-1] in TARGET_PROJECTIONS]


def _replace(model, full_name, new_module):
    parent_name, _, child_name = full_name.rpartition(".")
    parent = model.get_submodule(parent_name) if parent_name else model
    setattr(parent, child_name, new_module)


def quantize_model(base_model, cfg):
    q_model = copy.deepcopy(base_model)
    for name, lin in _target_linears(q_model):
        _replace(q_model, name, QuantizedLinear.from_float(
            lin, cfg["bits"], cfg["granularity"], cfg["symmetric"]))
    return q_model

Appending to quantizer_core.py


In [ ]:
%%writefile -a quantizer_core.py

def tensor_bytes(model, only_modules=None):
    """Calculate bytes used by all parameters and buffers, counting shared storage once."""
    seen, total = set(), 0
    mods = only_modules if only_modules is not None else [model]
    for mod in mods:
        for t in list(mod.parameters()) + list(mod.buffers()):
            key = (t.untyped_storage().data_ptr(), t.device)
            if key in seen:
                continue
            seen.add(key)
            total += t.numel() * t.element_size()
    return total


def linear_storage_bytes(model):
    """Calculate bytes held by the target projection matrices."""
    mods = [m for n, m in model.named_modules()
            if n.split(".")[-1] in TARGET_PROJECTIONS and isinstance(m, (nn.Linear, QuantizedLinear))]
    return tensor_bytes(model, mods), len(mods)


@torch.no_grad()
def compute_model_error(orig_model, q_model):
    """Compute overall MAE and RMSE over all quantized matrices in the model."""
    orig = dict(orig_model.named_modules())
    abs_sum = sq_sum = 0.0
    n = 0
    for name, ql in q_model.named_modules():
        if isinstance(ql, QuantizedLinear):
            w0 = orig[name].weight.data.to(torch.float32)
            diff = w0 - ql.dequantize_weight(torch.float32)
            abs_sum += diff.abs().sum().item()
            sq_sum += (diff ** 2).sum().item()
            n += w0.numel()
    return abs_sum / n, (sq_sum / n) ** 0.5


def export_checkpoint(model, out_dir, cfg=None, base_model_id=None, tokenizer=None):
    """Export model checkpoint to safetensors alongside configuration and tokenizer metadata."""
    from safetensors.torch import save_file
    os.makedirs(out_dir, exist_ok=True)
    sd = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}
    if getattr(model.config, "tie_word_embeddings", False):
        sd.pop("lm_head.weight", None)
    path = os.path.join(out_dir, "model.safetensors")
    save_file(sd, path, metadata={"format": "pt"})
    model.config.save_pretrained(out_dir)
    if cfg is not None:
        with open(os.path.join(out_dir, "quant_config.json"), "w") as f:
            json.dump({
                "format_version": 1,
                "base_model_id": base_model_id,
                "quantization": cfg,
                "target_projections": list(TARGET_PROJECTIONS)
            }, f, indent=2)
    if tokenizer is not None:
        tokenizer.save_pretrained(out_dir)
    return path


def load_quantized_model(path, device="cpu", dtype=torch.bfloat16):
    """Load a quantized model from checkpoint directory containing safetensors and metadata."""
    from safetensors.torch import load_file
    from transformers import AutoConfig, AutoModelForCausalLM
    with open(os.path.join(path, "quant_config.json")) as f:
        meta = json.load(f)
    cfg = meta["quantization"]
    config = AutoConfig.from_pretrained(path)
    model = AutoModelForCausalLM.from_config(config).to(dtype)
    for name, lin in _target_linears(model):
        _replace(model, name, QuantizedLinear(
            lin.in_features, lin.out_features, cfg["bits"], cfg["granularity"],
            cfg["symmetric"], lin.bias is not None))
    sd = load_file(os.path.join(path, "model.safetensors"))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if getattr(config, "tie_word_embeddings", False):
        model.tie_weights()
        missing = [k for k in missing if k != "lm_head.weight"]
    assert not unexpected and not missing, f"load problem: missing={missing}, unexpected={unexpected}"
    return model.to(device).eval()

Appending to quantizer_core.py


In [ ]:
importlib.reload(quantizer_core)
import quantizer_core as qz
from quantizer_core import *

CONFIGS = {
    "A: INT8 per-tensor (sym)":   dict(bits=8, granularity="tensor",  symmetric=True),
    "B: INT8 per-channel (sym)":  dict(bits=8, granularity="channel", symmetric=True),
    "C: INT4 per-channel (asym)": dict(bits=4, granularity="channel", symmetric=False),
}

# Quantization parameters on a real layer: block 0, q_proj
w32 = baseline_model.model.layers[0].self_attn.q_proj.weight.data.to(torch.float32)
print(f"model.layers.0.self_attn.q_proj {tuple(w32.shape)} | original min {w32.min().item():.5f}, max {w32.max().item():.5f}")
rows = []
for name, cfg in CONFIGS.items():
    r = quantize_weight(w32, cfg["bits"], cfg["granularity"], symmetric=cfg["symmetric"])
    rows.append({"config": name,
                 "min (row 0)": round(r["w_min"].flatten()[0].item(), 5),
                 "max (row 0)": round(r["w_max"].flatten()[0].item(), 5),
                 "q_min": r["q_min"], "q_max": r["q_max"],
                 "scale": round(r["scale"].flatten()[0].item(), 6),
                 "zero point": int(r["zero_point"].flatten()[0].item()),
                 "# scales": r["scale"].numel()})
display(pd.DataFrame(rows).set_index("config"))
print("(for the per-tensor config, 'row 0' is the whole matrix)")

# First 8 weights of row 0 under config C: original -> stored int4 -> reconstructed
r = quantize_weight(w32, 4, "channel", symmetric=False)
s0, z0, q8 = r["scale"][0, 0], r["zero_point"][0, 0], r["q"][0, :8]
print(f"\nConfig C, row 0 (scale {s0.item():.6f}, zero point {int(z0.item())})")
print("original      :", [round(v, 5) for v in w32[0, :8].tolist()])
print("stored int4   :", q8.tolist())
print("reconstructed :", [round(v, 5) for v in dequantize_tensor(q8, s0, z0).tolist()])
del r

model.layers.0.self_attn.q_proj (2048, 1024) | original min -0.58984, max 0.64453


,min (row 0),max (row 0),q_min,q_max,scale,zero point,# scales
config,,,,,,,
A: INT8 per-tensor (sym),-0.58984,0.64453,-128,127,0.005074,0,1
B: INT8 per-channel (sym),-0.05176,0.03979,-128,127,0.000407,0,2048
C: INT4 per-channel (asym),-0.05176,0.03979,-8,7,0.006104,0,2048


(for the per-tensor config, 'row 0' is the whole matrix)

Config C, row 0 (scale 0.006104, zero point 0)
original      : [0.0034, -0.00346, -0.0127, 0.02039, 0.01428, -0.00349, -0.00249, 0.02026]
stored int4   : [1, -1, -2, 3, 2, -1, 0, 3]
reconstructed : [0.0061, -0.0061, -0.01221, 0.01831, 0.01221, -0.0061, 0.0, 0.01831]


## Stage 6 — Evaluation setup
For each configuration we measure: **size** (weights in memory, projection matrices alone, saved file), **error** (MAE / RMSE over all 196 matrices), **quality** (WikiText-2 perplexity on the first 32,768 test tokens, window 512, plus two automatic task checks) and **speed** (greedy tokens/second after warm-up).

Quantized layers rebuild their weights on every forward pass, so they are expected to be *slower* than the baseline: the saving is memory and storage, not speed.

In [ ]:
SIZE_TARGET, MAX_PPL_INCREASE, USABLE_RAM_BYTES = 0.60, 0.10, 8e9
N_EVAL_TOKENS, PPL_WINDOW, GEN_TOKENS, SPEED_TOKENS = 32768, 512, 128, 64
LINEAR_PARAMS = sum(m.weight.numel() for _, m in qz._target_linears(baseline_model))

FALLBACK_TEXT = (
    "The river bends slowly through the valley before it reaches the sea. Farmers in the region plant wheat in autumn "
    "and harvest it in early summer, when the days are long and dry. A small library in the village keeps old maps, "
    "letters and ledgers that describe how the town grew over three centuries. Engineers measured the bridge twice, "
    "because a single error in the calculation could weaken the whole structure. Children learn to count by grouping "
    "objects, then by writing numbers, and finally by reasoning about patterns. In the evening the market fills with "
    "the smell of bread, spices and roasted coffee, and people stop to talk about the weather and the price of grain. ")

def load_eval_ids(tok, n_tokens):
    try:
        from datasets import load_dataset
        ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
        text, source = "\n\n".join(ds["text"])[:600000], "WikiText-2 test"
    except Exception as e:
        print(f"WARNING: WikiText-2 unavailable ({type(e).__name__}); using a tiny built-in text. "
              "Perplexities are not comparable with published numbers.")
        text, source = FALLBACK_TEXT * 40, "built-in fallback text"
    return tok(text, return_tensors="pt").input_ids[:, :n_tokens], source

eval_ids, eval_source = load_eval_ids(tokenizer, N_EVAL_TOKENS)
print(f"Perplexity data: {eval_source}, {eval_ids.shape[1]:,} tokens")

def sync():
    if cuda_available:
        torch.cuda.synchronize()

@torch.no_grad()
def perplexity(model, ids, window=PPL_WINDOW):
    model.eval()
    nll, count = 0.0, 0
    for i in range(0, ids.shape[1] - 1, window):
        chunk = ids[:, i:i + window].to(device)
        if chunk.shape[1] < 2:
            break
        logits = model(chunk).logits
        nll += F.cross_entropy(logits[0, :-1].float(), chunk[0, 1:], reduction="sum").item()
        count += chunk.shape[1] - 1
    return math.exp(nll / count)

def chat_inputs(prompt):
    msgs = [{"role": "user", "content": prompt}]
    try:
        t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tokenizer(t, return_tensors="pt").to(device)

@torch.no_grad()
def generate_text(model, prompt, max_new_tokens):
    inp = chat_inputs(prompt)
    out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def benchmark_speed(model, n_new=SPEED_TOKENS, repeats=2):
    inp = chat_inputs("Describe how a bicycle works.")
    model.generate(**inp, max_new_tokens=8, do_sample=False)                 # warm-up
    sync(); t0 = time.perf_counter()
    for _ in range(repeats):
        model.generate(**inp, max_new_tokens=n_new, min_new_tokens=n_new, do_sample=False)
    sync()
    return n_new * repeats / (time.perf_counter() - t0)

EVAL_PROMPTS = [
    "Explain why the sky is blue in three simple sentences.",
    "If a train travels at 60 mph for 2.5 hours, how many miles does it cover? Show your calculation.",
    "Write a short Python function that reverses a list without using the built-in reverse method.",
    "List 3 healthy breakfast ideas and explain why each is nutritious.",
]

def task_checks(responses):
    """Two automatic checks: the arithmetic answer (150) appears, and the code answer defines a function."""
    return int("150" in responses[1]) + int("def " in responses[2])

def evaluate_config(name, model, cfg=None):
    model.eval()
    total_bytes = qz.tensor_bytes(model)
    lin_bytes, _ = qz.linear_storage_bytes(model)
    ckpt_dir = os.path.join("checkpoints", "".join(ch if ch.isalnum() else "_" for ch in name))
    disk_bytes = os.path.getsize(qz.export_checkpoint(model, ckpt_dir))
    shutil.rmtree(ckpt_dir)
    mae, rmse = (0.0, 0.0) if cfg is None else qz.compute_model_error(baseline_model, model)
    responses = [generate_text(model, p, GEN_TOKENS) for p in EVAL_PROMPTS]
    return dict(name=name, cfg=cfg, total_bytes=total_bytes, lin_bytes=lin_bytes, disk_bytes=disk_bytes,
                mae=mae, rmse=rmse, ppl=perplexity(model, eval_ids), tok_s=benchmark_speed(model),
                checks=task_checks(responses), responses=responses)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Perplexity data: built-in fallback text, 5,201 tokens


## Stage 7 — Run all configurations
Each quantized model is built from the baseline, measured, then released to keep GPU memory low.

In [ ]:
results = [evaluate_config(f"Baseline ({BASE_NAME})", baseline_model)]
if not math.isfinite(results[0]["ppl"]):
    raise RuntimeError("Baseline perplexity is NaN/inf. Try a runtime with BF16 support.")
print(f"{results[0]['name']:<28} perplexity {results[0]['ppl']:.3f} | {results[0]['tok_s']:.1f} tok/s")

for name, cfg in CONFIGS.items():
    qm = qz.quantize_model(baseline_model, cfg)
    r = evaluate_config(name, qm, cfg)
    results.append(r)
    print(f"{name:<28} perplexity {r['ppl']:.3f} | {r['tok_s']:.1f} tok/s | {r['total_bytes'] / MB:,.0f} MB")
    del qm
    gc.collect()
    if cuda_available:
        torch.cuda.empty_cache()

Baseline (BF16)              perplexity 2.687 | 19.2 tok/s
A: INT8 per-tensor (sym)     perplexity 2.681 | 15.7 tok/s | 752 MB
B: INT8 per-channel (sym)    perplexity 2.689 | 8.1 tok/s | 752 MB
C: INT4 per-channel (asym)   perplexity 3.588 | 6.5 tok/s | 533 MB


## Stage 8 — Results
Every value is computed from the measured objects. `results.csv` holds the same data for the report.

In [ ]:
base = results[0]
for r in results:
    r["red_fp32"]      = 1 - r["total_bytes"] / FP32_BYTES
    r["red_base"]      = 1 - r["total_bytes"] / base["total_bytes"]
    r["lin_red"]       = 1 - r["lin_bytes"] / base["lin_bytes"]
    r["bits_per_w"]    = r["lin_bytes"] * 8 / LINEAR_PARAMS
    r["ppl_inc"]       = r["ppl"] / base["ppl"] - 1
    r["fits_ram"]      = r["total_bytes"] < USABLE_RAM_BYTES
    r["meets_size"]    = r["red_fp32"] >= SIZE_TARGET
    r["meets_quality"] = r["ppl_inc"] <= MAX_PPL_INCREASE

yn = lambda b: "yes" if b else "NO"

size_df = pd.DataFrame([{
    "Config": r["name"],
    "Projection weights (MB)": f"{r['lin_bytes'] / MB:,.1f}",
    "Bits per weight": f"{r['bits_per_w']:.2f}",
    "Total in memory (MB)": f"{r['total_bytes'] / MB:,.1f}",
    "File on disk (MB)": f"{r['disk_bytes'] / MB:,.1f}",
    "Reduction vs FP32": f"{r['red_fp32'] * 100:.1f}%",
    f"Reduction vs {BASE_NAME}": f"{r['red_base'] * 100:.1f}%",
    f"Projections only vs {BASE_NAME}": f"{r['lin_red'] * 100:.1f}%",
    "Fits in 8 GB": yn(r["fits_ram"]),
} for r in results]).set_index("Config")

quality_df = pd.DataFrame([{
    "Config": r["name"],
    "Weight MAE": f"{r['mae']:.5f}",
    "Weight RMSE": f"{r['rmse']:.5f}",
    "Perplexity": f"{r['ppl']:.3f}",
    "Perplexity change": "-" if r is base else f"{r['ppl_inc'] * 100:+.1f}%",
    "Speed (tok/s)": f"{r['tok_s']:.1f}",
    "Task checks (of 2)": r["checks"],
    f"Size >= {SIZE_TARGET * 100:.0f}% vs FP32": "-" if r is base else yn(r["meets_size"]),
    f"Perplexity within +{MAX_PPL_INCREASE * 100:.0f}%": "-" if r is base else yn(r["meets_quality"]),
} for r in results]).set_index("Config")

print(f"Size (FP32 reference = {FP32_BYTES / MB:,.0f} MB)")
display(size_df)
print(f"Quality and speed ({eval_source})")
display(quality_df)
pd.concat([size_df, quality_df], axis=1).to_csv("results.csv")

Size (FP32 reference = 2,384 MB)


,Projection weights (MB),Bits per weight,Total in memory (MB),File on disk (MB),Reduction vs FP32,Reduction vs BF16,Projections only vs BF16,Fits in 8 GB
Config,,,,,,,,
Baseline (BF16),880.8,16.00,"1,192.1","1,192.1",50.0%,0.0%,0.0%,yes
A: INT8 per-tensor (sym),440.4,8.00,751.7,751.8,68.5%,36.9%,50.0%,yes
B: INT8 per-channel (sym),441.1,8.01,752.4,752.4,68.4%,36.9%,49.9%,yes
C: INT4 per-channel (asym),221.2,4.02,532.5,532.6,77.7%,55.3%,74.9%,yes


Quality and speed (built-in fallback text)


,Weight MAE,Weight RMSE,Perplexity,Perplexity change,Speed (tok/s),Task checks (of 2),Size >= 60% vs FP32,Perplexity within +10%
Config,,,,,,,,
Baseline (BF16),0.00000,0.00000,2.687,-,19.2,2,-,-
A: INT8 per-tensor (sym),0.00086,0.00108,2.681,-0.2%,15.7,2,yes,yes
B: INT8 per-channel (sym),0.00021,0.00026,2.689,+0.1%,8.1,2,yes,yes
C: INT4 per-channel (asym),0.00339,0.00410,3.588,+33.6%,6.5,2,yes,NO


## Stage 9 — Winner
Applying the selection rule from the top of the notebook.

In [ ]:
cands = [r for r in results[1:] if r["meets_size"]]
good  = [r for r in cands if r["meets_quality"]]
if good:
    smallest = min(r["total_bytes"] for r in good)
    near = [r for r in good if r["total_bytes"] <= smallest * 1.01]
    winner = min(near, key=lambda r: r["ppl"])
    why = "smallest configuration meeting both the size target and the perplexity bound"
elif cands:
    winner = min(cands, key=lambda r: r["ppl"])
    why = "none met the perplexity bound, so this is the lowest-perplexity configuration meeting the size target"
else:
    winner = max(results[1:], key=lambda r: r["red_fp32"])
    why = "WARNING: none met the size target; this has the largest reduction"

for r in results[1:]:
    print(f"{r['name']:<28} size {r['red_fp32'] * 100:5.1f}% vs FP32 ({yn(r['meets_size'])}) | "
          f"perplexity {r['ppl_inc'] * 100:+6.1f}% ({yn(r['meets_quality'])}) | MAE {r['mae']:.5f}")

print(f"\nWinner: {winner['name']}  ({why})")
print(f"  Size    : {winner['total_bytes'] / MB:,.0f} MB in memory, {winner['red_fp32'] * 100:.1f}% smaller than FP32 and "
      f"{winner['red_base'] * 100:.1f}% smaller than {BASE_NAME} ({winner['lin_red'] * 100:.1f}% on the projection matrices; "
      "the embedding/output matrix stays 16-bit)")
print(f"  Quality : perplexity {winner['ppl']:.3f} vs {base['ppl']:.3f} baseline ({winner['ppl_inc'] * 100:+.1f}%), "
      f"MAE {winner['mae']:.5f}, task checks {winner['checks']}/2")
print(f"  Speed   : {winner['tok_s']:.1f} tok/s vs {base['tok_s']:.1f} baseline (weights are rebuilt on every forward pass)")

A: INT8 per-tensor (sym)     size  68.5% vs FP32 (yes) | perplexity   -0.2% (yes) | MAE 0.00086
B: INT8 per-channel (sym)    size  68.4% vs FP32 (yes) | perplexity   +0.1% (yes) | MAE 0.00021
C: INT4 per-channel (asym)   size  77.7% vs FP32 (yes) | perplexity  +33.6% (NO) | MAE 0.00339

Winner: A: INT8 per-tensor (sym)  (smallest configuration meeting both the size target and the perplexity bound)
  Size    : 752 MB in memory, 68.5% smaller than FP32 and 36.9% smaller than BF16 (50.0% on the projection matrices; the embedding/output matrix stays 16-bit)
  Quality : perplexity 2.681 vs 2.687 baseline (-0.2%), MAE 0.00086, task checks 2/2
  Speed   : 15.7 tok/s vs 19.2 baseline (weights are rebuilt on every forward pass)


## Stage 10 — Final model: export, reload, verify
The winning configuration is rebuilt and saved (`model.safetensors`, `quant_config.json`, tokenizer, `quantizer_core.py`), then **reloaded from disk** to confirm it behaves exactly like the in-memory model.

In [ ]:
final_name, final_cfg = winner["name"], winner["cfg"]
FINAL_DIR = "final_quantized_model"
if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)

final_model = qz.quantize_model(baseline_model, final_cfg).eval()
qz.export_checkpoint(final_model, FINAL_DIR, cfg=final_cfg, base_model_id=MODEL_ID, tokenizer=tokenizer)
shutil.copy("quantizer_core.py", os.path.join(FINAL_DIR, "quantizer_core.py"))

reloaded = qz.load_quantized_model(FINAL_DIR, device=device, dtype=dtype)
with torch.no_grad():
    probe = eval_ids[:, :256].to(device)
    same_logits = torch.equal(final_model(probe).logits, reloaded(probe).logits)
ppl_a, ppl_b = perplexity(final_model, eval_ids), perplexity(reloaded, eval_ids)
same_text = generate_text(final_model, EVAL_PROMPTS[1], GEN_TOKENS) == generate_text(reloaded, EVAL_PROMPTS[1], GEN_TOKENS)
n_layers = sum(isinstance(m, QuantizedLinear) for m in reloaded.modules())
mb = os.path.getsize(os.path.join(FINAL_DIR, "model.safetensors")) / MB

print(f"Final model: {final_name}")
print("Saved in '" + FINAL_DIR + "/': " + ", ".join(f"{f} ({os.path.getsize(os.path.join(FINAL_DIR, f)) / MB:,.2f} MB)" for f in sorted(os.listdir(FINAL_DIR))))
print(f"model.safetensors: {mb:,.0f} MB, {(1 - mb * MB / FP32_BYTES) * 100:.1f}% smaller than FP32, "
      f"{(1 - mb * MB / base['disk_bytes']) * 100:.1f}% smaller than the {BASE_NAME} file")
print(f"Reload check: {n_layers} quantized layers | logits identical: {same_logits} | "
      f"perplexity {ppl_a:.4f} vs {ppl_b:.4f} | greedy output identical: {same_text}")
del reloaded

Final model: A: INT8 per-tensor (sym)
Saved in 'final_quantized_model/': chat_template.jinja (0.00 MB), config.json (0.00 MB), model.safetensors (751.75 MB), quant_config.json (0.00 MB), quantizer_core.py (0.01 MB), tokenizer.json (11.42 MB), tokenizer_config.json (0.00 MB)
model.safetensors: 752 MB, 68.5% smaller than FP32, 36.9% smaller than the BF16 file
Reload check: 196 quantized layers | logits identical: False | perplexity 2.6813 vs 2.6817 | greedy output identical: True
